In [1]:
import os
import dlt
from dlt.sources.helpers import requests      # requests with automatic retries
from lake import PIPELINES_DIR, lake_destination, add_metadata, list_lake, read_lake

DOG_API_URL = os.getenv("DOG_API_URL", "https://dog.ceo/api/breeds/list/all")

In [2]:
@dlt.resource(name="breeds", write_disposition="replace")
def dog_breeds():
    response = requests.get(DOG_API_URL, timeout=30)
    response.raise_for_status()                 # stop if the API answers with an error
    for breed, sub_breeds in response.json()["message"].items():
        if not sub_breeds:                      # breed without sub-breeds, e.g. "akita"
            yield {"breed": breed, "sub_breed": None, "full_name": breed,
                   "images_api": f"https://dog.ceo/api/breed/{breed}/images"}
        for sub in sub_breeds:                  # one row per sub-breed, e.g. "australian shepherd"
            yield {"breed": breed, "sub_breed": sub, "full_name": f"{breed} {sub}",
                   "images_api": f"https://dog.ceo/api/breed/{breed}/{sub}/images"}


def build_api_resource():
    return add_metadata(dog_breeds(), source_system="dogceo",
                        source_resource="breeds/list/all", ingestion_type="full")

In [3]:
api_pipeline = dlt.pipeline(
    pipeline_name="dogceo_full",
    destination=lake_destination("silo"),
    dataset_name="dogceo",
    pipelines_dir=PIPELINES_DIR,
)
api_pipeline.run(build_api_resource(), loader_file_format="parquet")
print("Rows:", api_pipeline.last_trace.last_normalize_info.row_counts)
api_keys = list_lake("dogceo/breeds")
api_keys

Rows: {'breeds': 165, '_dlt_pipeline_state': 1}


,key,size_kb,last_modified
0,dogceo/breeds/ingestion_date=2026-10-29/179330...,12.3,2026-10-29 20:43:53.404+00:00


In [4]:
breeds = read_lake("dogceo/breeds/")
print(len(breeds), "rows")
breeds.head(10)

165 rows


,breed,full_name,images_api,_source_system,_source_resource,_ingestion_type,_ingested_at,_load_id,_dlt_load_id,_dlt_id,sub_breed
0,affenpinscher,affenpinscher,https://dog.ceo/api/breed/affenpinscher/images,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,b00Oid2tJlny6g,NaN
1,african,african wild,https://dog.ceo/api/breed/african/wild/images,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,M4AHuiG+XAF5bQ,wild
2,airedale,airedale,https://dog.ceo/api/breed/airedale/images,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,iafQg7y0o4ETvg,NaN
3,akita,akita,https://dog.ceo/api/breed/akita/images,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,/EoKGVKadrJZIg,NaN
4,appenzeller,appenzeller,https://dog.ceo/api/breed/appenzeller/images,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,LP/IJUJOH5w54g,NaN
5,australian,australian kelpie,https://dog.ceo/api/breed/australian/kelpie/im...,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,xNJNcLWQHqXycA,kelpie
6,australian,australian shepherd,https://dog.ceo/api/breed/australian/shepherd/...,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,zf7unIH9+lMjCA,shepherd
7,bakharwal,bakharwal indian,https://dog.ceo/api/breed/bakharwal/indian/images,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,v6dlRgnMNIXJWw,indian
8,basenji,basenji,https://dog.ceo/api/breed/basenji/images,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,GdangdOKtj1qBQ,NaN
9,beagle,beagle,https://dog.ceo/api/breed/beagle/images,dogceo,breeds/list/all,full,2026-10-29 20:43:51.486947+00:00,1793306631.711786,1793306631.711786,zAHOeaurCPjOcQ,NaN


In [5]:
before = set(api_keys.key)
api_pipeline.run(build_api_resource(), loader_file_format="parquet")
after = set(list_lake("dogceo/breeds").key)
print("REMOVED:", sorted(before - after))
print("ADDED:  ", sorted(after - before))
print("Rows now:", len(read_lake("dogceo/breeds/")))

REMOVED: ['dogceo/breeds/ingestion_date=2026-10-29/1793306631.711786.8f92a2902d.parquet']
ADDED:   ['dogceo/breeds/ingestion_date=2026-10-29/1793306658.8310988.4650208d6a.parquet']
Rows now: 165


In [6]:
shelter_dogs = read_lake("postgres/animals/")
shelter_dogs = shelter_dogs[shelter_dogs.species == "dog"]
known = set(breeds.breed.str.lower())
shelter_dogs["breed_in_api"] = shelter_dogs.breed.str.lower().str.split().str[-1].isin(known)
print(f"{shelter_dogs.breed_in_api.mean():.0%} of shelter dogs have a breed word that Dog CEO knows")

44% of shelter dogs have a breed word that Dog CEO knows


**Conclusion:** the Dog CEO API (dog breeds) fits PawHome because our animals have breeds.
It is loaded with mode `replace` into `dogceo/breeds/ingestion_date=…/…parquet`, with the same
metadata columns. On a rerun the old file is replaced by a new one with a new load id; the
row count only changes if the API's data changes. Having both sources in one lake lets us
combine them (cell 6).